In [1]:
from __future__ import print_function, division

import os
import warnings
import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import cv2

warnings.filterwarnings("ignore")

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
torch.backends.cudnn.benchmark = True

model_full_name = "efficientnet-b4-e10"
model_name = "efficientnet-b4"
folder_name = "effnetmodelb4"

TEST_DIR = "/kaggle/input/cassava-leaf-disease-classification/test_images/"
PATH = "/kaggle/input/{}/{}.pt".format(folder_name, model_full_name)

sample_path = "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv"



In [2]:
from torchvision.models import EfficientNet_B4_Weights
from torchvision.transforms.functional import to_pil_image

try:
    backbone_weights = EfficientNet_B4_Weights.IMAGENET1K_V1
except Exception:
    backbone_weights = None

if backbone_weights is not None:
    tv_transform = backbone_weights.transforms()
    mean = tuple(float(x) for x in tv_transform.mean)
    std = tuple(float(x) for x in tv_transform.std)
    resize_size = getattr(tv_transform, "resize_size", None)
    crop_size = getattr(tv_transform, "crop_size", None)
else:
    tv_transform = None
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
    resize_size, crop_size = None, None

if os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    test_ids = sample["image_id"].tolist()
else:
    test_ids = None

print("Using device:", device)
print("Using backbone ImageNet weights available:", backbone_weights is not None)
print("Torchvision weights mean/std:", mean, std)
print("Torchvision weights resize_size:", resize_size, "crop_size:", crop_size)




Using device: cuda:0
Using backbone ImageNet weights available: True
Torchvision weights mean/std: (0.485, 0.456, 0.406) (0.229, 0.224, 0.225)
Torchvision weights resize_size: [384] crop_size: [380]


In [3]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None, image_list=None):
        self.root_dir = root_dir
        self.transform = transform

        if image_list is not None:
            self.images = list(image_list)
        else:
            self.images = sorted(
                [f for f in os.listdir(root_dir) if f.lower().endswith(".jpg")]
            )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_name = self.images[idx]
        img_path = os.path.join(self.root_dir, img_name)

        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise IOError("Failed to read image: {}".format(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if image.ndim == 2:
            image = np.stack([image, image, image], axis=-1)
        elif image.shape[2] == 4:
            image = image[:, :, :3]
        image = np.ascontiguousarray(image, dtype=np.uint8)

        if self.transform is not None:
            pil_img = to_pil_image(image)  # RGB PIL.Image
            image = self.transform(pil_img)  # tensor CxHxW float normalized

        return img_name, image


test_image = TestDataset(root_dir=TEST_DIR, transform=tv_transform, image_list=test_ids)

testloader = DataLoader(
    test_image, batch_size=8, shuffle=False, num_workers=4, pin_memory=use_cuda
)

print("Test images:", len(test_image))



Test images: 2676


In [4]:
from torchvision.models import efficientnet_b4

model = efficientnet_b4(weights=backbone_weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)
model = model.to(device)


def _unwrap_state_dict(obj):
    if isinstance(obj, dict):
        if len(obj) > 0:
            any_v = next(iter(obj.values()))
            if torch.is_tensor(any_v):
                return obj
        for key in [
            "state_dict",
            "model_state_dict",
            "model",
            "net",
            "network",
            "module",
        ]:
            if key in obj and isinstance(obj[key], dict):
                return _unwrap_state_dict(obj[key])
    return obj


def find_checkpoint_path_strict(preferred_path):
    if os.path.exists(preferred_path):
        return preferred_path

    base_pt = os.path.basename(preferred_path)
    base_noext = os.path.splitext(base_pt)[0]

    patterns = [
        "/kaggle/input/**/{}".format(base_pt),
        "/kaggle/input/**/{}.pth".format(base_noext),
        "/kaggle/input/**/{}*.pt".format(base_noext),
        "/kaggle/input/**/{}*.pth".format(base_noext),
        "/kaggle/input/{}/{}.pt".format(folder_name, model_full_name),
        "/kaggle/input/{}/{}.pth".format(folder_name, model_full_name),
        # Change (score improvement): broaden search to common naming for Cassava fine-tuned EfficientNet-B4 checkpoints
        # so we more reliably load the intended fine-tuned weights instead of falling back to ImageNet.
        "/kaggle/input/**/*b4*.pt",
        "/kaggle/input/**/*b4*.pth",
        "/kaggle/input/**/*efficientnet*b4*.pt",
        "/kaggle/input/**/*efficientnet*b4*.pth",
    ]

    candidates = []
    for pat in patterns:
        candidates.extend(glob.glob(pat, recursive=True))

    candidates = sorted(set(candidates))
    if not candidates:
        return None

    def score_path(p):
        p_low = p.lower()
        bn = os.path.basename(p_low)
        exact = 0 if bn in (base_pt.lower(), (base_noext + ".pth").lower()) else 1

        # Heuristics: prefer paths that look cassava/leaf-related and contain "b4".
        # This is still minimal and only affects which checkpoint we load.
        cassava_hint = 0 if ("cassava" in p_low or "leaf" in p_low) else 1
        b4_hint = 0 if ("b4" in p_low) else 1
        eff_hint = 0 if ("efficientnet" in p_low or "effnet" in p_low) else 1

        # Prefer shorter paths when ties remain.
        return (exact, cassava_hint, b4_hint, eff_hint, len(p_low), p_low)

    candidates = sorted(candidates, key=score_path)
    return candidates[0]


def _strip_known_prefixes(k):
    for pref in ["module.", "model.", "net.", "network."]:
        if k.startswith(pref):
            return k[len(pref) :]
    return k


def _remap_classifier_keys_if_needed(sd, model_obj):
    if not isinstance(sd, dict):
        return sd

    model_sd = model_obj.state_dict()
    out = dict(sd)

    def try_map(src_w, src_b, dst_w, dst_b):
        if src_w in out and dst_w in model_sd:
            if tuple(out[src_w].shape) == tuple(model_sd[dst_w].shape):
                out[dst_w] = out[src_w]
        if src_b in out and dst_b in model_sd:
            if tuple(out[src_b].shape) == tuple(model_sd[dst_b].shape):
                out[dst_b] = out[src_b]

    try_map(
        "classifier.weight",
        "classifier.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )
    try_map(
        "model.classifier.weight",
        "model.classifier.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )
    try_map(
        "model.classifier.1.weight",
        "model.classifier.1.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )
    try_map(
        "net.classifier.1.weight",
        "net.classifier.1.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )

    try_map(
        "classifier.1.1.weight",
        "classifier.1.1.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )
    try_map(
        "model.classifier.1.1.weight",
        "model.classifier.1.1.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )
    try_map(
        "net.classifier.1.1.weight",
        "net.classifier.1.1.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )

    try_map(
        "classifier.1.weight",
        "classifier.1.bias",
        "classifier.1.weight",
        "classifier.1.bias",
    )

    return out


def _classifier_loaded_ok(model_obj, loaded_keys_set):
    # Change (score improvement): instead of a brittle "loaded fraction" based on missing_keys,
    # directly verify the 5-class classifier weights/bias loaded (critical for good accuracy).
    needed = {"classifier.1.weight", "classifier.1.bias"}
    return needed.issubset(set(loaded_keys_set))


ckpt_path = find_checkpoint_path_strict(PATH)
if ckpt_path is None:
    print(
        "WARNING: No fine-tuned checkpoint found; running with ImageNet weights only."
    )
else:
    print("Loading checkpoint:", ckpt_path)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state_dict = _unwrap_state_dict(ckpt)

    if isinstance(state_dict, dict):
        new_sd = {}
        for k, v in state_dict.items():
            nk = _strip_known_prefixes(k)
            new_sd[nk] = v
        state_dict = new_sd

        state_dict = _remap_classifier_keys_if_needed(state_dict, model)

        # Track which keys we are actually providing to load_state_dict.
        provided_keys = list(state_dict.keys())

        missing, unexpected = model.load_state_dict(state_dict, strict=False)

        print("Checkpoint load_state_dict(strict=False) summary:")
        print("  Provided keys:", len(provided_keys))
        print("  Missing keys:", len(missing))
        print("  Unexpected keys:", len(unexpected))
        if len(missing) > 0:
            print("  Missing keys (first 10):", missing[:10])
        if len(unexpected) > 0:
            print("  Unexpected keys (first 10):", unexpected[:10])

        # Change (score improvement): hard-fail only if classifier weights were not loaded,
        # because that strongly indicates we're not using the fine-tuned 5-class head.
        if not _classifier_loaded_ok(model, provided_keys):
            raise RuntimeError(
                "Checkpoint found at '{}' but did not provide classifier.1.(weight|bias) "
                "after prefix stripping/remapping; this likely means an incompatible checkpoint.".format(
                    ckpt_path
                )
            )
    else:
        print(
            "WARNING: Checkpoint format not recognized as state_dict; using ImageNet weights only."
        )

model = model.to(device)
model.eval()



Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:00<00:00, 96.1MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
            (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [5]:
names = []
predicted = []

with torch.no_grad():
    for names_batch, images_batch in testloader:
        images_batch = images_batch.to(device, non_blocking=use_cuda)
        output = model(images_batch)
        pred = torch.argmax(output, dim=1).cpu().numpy()
        names.extend(list(names_batch))
        predicted.extend(list(pred))

print("Predicted:", len(predicted))



Predicted: 2676


In [6]:
result = pd.DataFrame({"image_id": names, "label": predicted})

if os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    result = sample[["image_id"]].merge(result, on="image_id", how="left")
    result["label"] = result["label"].fillna(0).astype(int)
else:
    result["label"] = result["label"].astype(int)

result.to_csv("submission.csv", index=False)
print(result.head())
print("Wrote submission.csv with {} rows".format(len(result)))

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      2
2  1234375577.jpg      2
3  1234555380.jpg      2
4  1234571117.jpg      2
Wrote submission.csv with 2676 rows
